<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/04_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Data Preparation: Model A, 1-minute data (t+3h storm forecast)

Builds training/validation/test windows from omni_1m_clean.parquet
(1-minute solar wind, 1995-present) with labels from the hourly
omni_clean.csv (1-minute files do not carry Kp).

- **Physics at full resolution**: coupling features (Newell, rectified
  speed, pressure) computed on 1-minute values, then averaged into 5-min
  bins — this preserves the sharp shock/southward-turning structure that
  hourly averaging erased
- **Sample** = 24h window of 5-min bins (288 x 6 features) ending at
  hour *t*, labeled with the exact Kp of the next 3-hour block (same
  convention as the hourly track)
- **Coverage rule**: windows need &gt;=95% valid 5-min bins (recent years have
  ~30% missing minutes); gaps &lt;=30 min are interpolated
- **Splits**: time-ordered: train 1995-2010, val 2011-2018, test 2019+
- **Output**: windows_a1m.npz + norm json (large; kept on Drive, not in git)

In [5]:
import pandas as pd
import numpy as np
import json
from numpy.lib.stride_tricks import sliding_window_view

PARQUET = '/content/drive/MyDrive/temp/omni_1m_clean.parquet'
CSV     = '/content/drive/MyDrive/temp/omni_clean.csv'
OUT_DIR = '/content/drive/MyDrive/temp'

## 1-minute physics features -> 5-min bins

In [6]:
m = pd.read_parquet(PARQUET)
print(f"1-min rows: {len(m)}  {m.index[0]} -> {m.index[-1]}")

# physics features at full 1-minute resolution (NaN propagates from inputs)
bt, by, bz = m['b_magnitude'], m['by_gsm'], m['bz_gsm']
v, n = m['speed'], m['density']
clock = np.degrees(np.arctan2(by, bz)) % 360
south = np.sin(np.radians(clock) / 2) ** (8 / 3)

m['newell']  = v**(4/3) * np.maximum(bt, 0)**(2/3) * south
m['rect_v']  = v * np.clip(-bz, 0, None)
m['pressure'] = n * v**2

# 5-min bins: extrema for coupling channels (peak driving), means for context
F = m[['bz_gsm', 'speed', 'density', 'newell', 'rect_v', 'pressure']] \
      .resample('5min').agg({'bz_gsm': 'min', 'speed': 'mean',
                             'density': 'mean', 'newell': 'max',
                             'rect_v': 'max', 'pressure': 'max'})
F = F.interpolate(limit=6, limit_direction='both')
print(f"5-min bins: {len(F)}  ({F.isna().any(axis=1).mean():.2%} still incomplete)")

FEATS = ['bz_gsm', 'speed', 'density', 'newell', 'rect_v', 'pressure']

1-min rows: 16830720  1995-01-01 00:00:00 -> 2026-12-31 23:59:00
5-min bins: 3366144  (3.46% still incomplete)


## Hourly windows (288 x 6) with exact next-block Kp labels

Window ending at hour t covers bins (t-24h, t]. Label = Kp of the next
3-hour block boundary at or after t (identical convention to the hourly
track; lead time always 0-3h).

In [7]:
data = F[FEATS].values.astype(np.float32)          # (n_bins, 6)
valid = ~np.isnan(data).any(axis=1)               # per-bin validity

W = sliding_window_view(data, (288, len(FEATS))).squeeze(1)   # stride-1 views
W = W[::12]                                        # one window per hour
vm = sliding_window_view(valid.astype(np.float32), 288)[::12]
vfrac = vm.mean(axis=1)

# window right edges: bins are START-stamped, so the last bin's start + 5min
# is the true end -> hour-aligned timestamps that exist in the hourly index
win_end = F.index[287::12] + pd.Timedelta(minutes=5)
assert win_end.equals(win_end.floor('h')), 'window ends not hour-aligned'

# labels from the hourly Kp series (next block boundary)
hr = pd.read_csv(CSV, parse_dates=['timestamp']).set_index('timestamp')
kp = hr['kp'].clip(upper=9).values
h = hr.index.hour.values
next_off = (3 - (h % 3)) % 3
next_off[next_off == 0] = 3
lidx = np.arange(len(hr)) + next_off
kp_next = np.full(len(hr), np.nan)
okk = lidx < len(hr)
kp_next[okk] = kp[lidx[okk]]
kp_next = pd.Series(kp_next, index=hr.index)

y = kp_next.reindex(win_end).values
assert not np.isnan(y).all(), 'labels failed to match the hourly index'

keep = (vfrac >= 0.95) & ~np.isnan(y)
X, y, t = W[keep], y[keep], win_end[keep]
print(f"windows kept: {len(X)} / {len(W)} "
      f"(coverage rule + label availability)")

windows kept: 242797 / 280489 (coverage rule + label availability)


## Splits and normalization

Train 1995-2010, val 2011-2018, test 2019-present. Normalization uses
train statistics only. Remaining NaNs (rare after interpolation) are set
to 0 *after* normalization, i.e. replaced by the training mean — a
neutral value, documented here rather than hidden.

In [8]:
tr = t <  '2011-01-01'
va = (t >= '2011-01-01') & (t < '2019-01-01')
te = t >= '2019-01-01'

flat = X[tr].reshape(-1, len(FEATS))
mu, sd = np.nanmean(flat, axis=0), np.nanstd(flat, axis=0)
sd = np.where(sd < 1e-6, 1.0, sd)

def sc(x):
    return np.nan_to_num((x - mu) / sd)          # NaN -> 0 == training mean

XA = dict(Xtr=sc(X[tr]), Xva=sc(X[va]), Xte=sc(X[te]),
          ytr=y[tr], yva=y[va], yte=y[te],
          ttr=t[tr], tva=t[va], tte=t[te])

for part in ['tr', 'va', 'te']:
    yy = XA['y' + part]
    print(f"{part:3s}: n={len(yy):6d}   "
          f"G1+ share (Kp>=5): {(yy >= 5).mean():.3f}   "
          f"Kp>=6 share: {(yy >= 6).mean():.3f}")

np.savez_compressed(f'{OUT_DIR}/windows_a1m.npz', **XA)
with open(f'{OUT_DIR}/windows_a1m_norm.json', 'w') as f:
    json.dump({'mean': mu.tolist(), 'std': sd.tolist(), 'features': FEATS},
              f, indent=1)
print('saved windows_a1m.npz + windows_a1m_norm.json')

tr : n=121277   G1+ share (Kp>=5): 0.034   Kp>=6 share: 0.011
va : n= 62258   G1+ share (Kp>=5): 0.022   Kp>=6 share: 0.006
te : n= 59262   G1+ share (Kp>=5): 0.024   Kp>=6 share: 0.007
saved windows_a1m.npz + windows_a1m_norm.json
